### GCRL implementation

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces


class FourRoomsMaze2D(gym.Env):
    """
    Continuous four-rooms. Agent is a point with position (x, y).
    Actions are small (dx, dy) displacements — many steps to cross a room.
    Obs: [x, y]  |  Action: [dx, dy] in [-1, 1], scaled by step_size
    """

    def __init__(
        self,
        maze_size: float       = 10.0,
        wall_thickness: float  = 0.4,
        door_width: float      = 1.0,
        step_size: float       = 0.1,   # max displacement per action
        max_steps: int         = 2000,
        render_mode            = None,
    ):
        super().__init__()
        self.maze_size      = maze_size
        self.wall_thickness = wall_thickness
        self.door_width     = door_width
        self.step_size      = step_size
        self.max_steps      = max_steps
        self.render_mode    = render_mode

        self.walls = self._build_walls()

        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0], dtype=np.float32),
            high=np.array([maze_size, maze_size], dtype=np.float32),
        )
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        )

        pad = wall_thickness + 0.5
        self.start_pos = np.array([pad,            maze_size - pad], dtype=np.float32)
        self.goal_pos  = np.array([maze_size - pad, pad],            dtype=np.float32)
        self.goal_radius = 0.5

        self._pos = self.start_pos.copy()
        self._step_count = 0
        self.np_random = np.random.default_rng()

    def _build_walls(self):
        s, wt, dw = self.maze_size, self.wall_thickness, self.door_width
        hw, mid   = wt / 2, s / 2
        walls = []

        # Outer boundary
        walls += [
            (0,      0,      s,      wt),
            (0,      s - wt, s,      s),
            (0,      0,      wt,     s),
            (s - wt, 0,      s,      s),
        ]

        # Horizontal divider with 2 doorways
        ldx, rdx = s * 0.25, s * 0.75
        walls += [
            (wt,            mid - hw,  ldx - dw/2,   mid + hw),
            (ldx + dw/2,    mid - hw,  rdx - dw/2,   mid + hw),
            (rdx + dw/2,    mid - hw,  s - wt,        mid + hw),
        ]

        # Vertical divider with 2 doorways
        tdy, bdy = s * 0.75, s * 0.25
        walls += [
            (mid - hw, wt,           mid + hw, bdy - dw/2),
            (mid - hw, bdy + dw/2,   mid + hw, tdy - dw/2),
            (mid - hw, tdy + dw/2,   mid + hw, s - wt),
        ]

        return [np.array(w, dtype=np.float32) for w in walls]

    def _resolve_collision(self, new_pos):
        pos = new_pos.copy()
        for (x0, y0, x1, y1) in self.walls:
            if x0 < pos[0] < x1 and y0 < pos[1] < y1:
                overlaps = [pos[0]-x0, x1-pos[0], pos[1]-y0, y1-pos[1]]
                min_idx = int(np.argmin(overlaps))
                if   min_idx == 0: pos[0] = x0
                elif min_idx == 1: pos[0] = x1
                elif min_idx == 2: pos[1] = y0
                else:              pos[1] = y1
        return pos

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.np_random = np.random.default_rng(seed)
        self._pos = self.start_pos.copy()
        self._step_count = 0
        return self._pos.copy(), {"pos": self._pos.tolist()}

    def step(self, action):
        action   = np.clip(action, -1.0, 1.0).astype(np.float32)
        new_pos  = self._pos + action * self.step_size
        new_pos  = np.clip(new_pos, 0.0, self.maze_size)
        self._pos = self._resolve_collision(new_pos)
        self._step_count += 1

        dist         = float(np.linalg.norm(self._pos - self.goal_pos))
        goal_reached = dist < self.goal_radius
        truncated    = self._step_count >= self.max_steps

        return (
            self._pos.copy(),
            1.0 if goal_reached else 0.0,
            bool(goal_reached),
            truncated,
            {"pos": self._pos.tolist(), "dist_to_goal": dist},
        )


class CustomFourRoomsMaze2D(gym.Wrapper):
    """Stochastic wrapper — Gaussian positional noise, mirrors CustomMountainCar."""

    def __init__(self, env, goal_reward=1.0, step_reward=0.0, wind_std=0.02):
        super().__init__(env)
        self.goal_reward = goal_reward
        self.step_reward = step_reward
        self.wind_std    = wind_std

    def step(self, action):
        rng  = self.unwrapped.np_random
        wind = rng.normal(0.0, self.wind_std, size=2).astype(np.float32) \
               if self.wind_std > 0 else np.zeros(2, dtype=np.float32)

        # Inject wind by perturbing action before base step
        noisy_action = np.clip(action + wind, -1.0, 1.0)
        obs, _, terminated, truncated, info = self.env.step(noisy_action)

        dist         = float(np.linalg.norm(
            self.unwrapped._pos - self.unwrapped.goal_pos))
        goal_reached = dist < self.unwrapped.goal_radius
        reward       = self.goal_reward if goal_reached else self.step_reward

        info = dict(info)
        info["wind_force"] = wind.tolist()
        return obs, reward, bool(goal_reached), truncated, info

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def render_maze2d(env, ax=None, trajectory=None):
    base = env.unwrapped
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.clear()

    s = base.maze_size
    ax.set_xlim(0, s)
    ax.set_ylim(0, s)
    ax.set_aspect("equal")
    ax.set_facecolor("#f0ede8")
    ax.axis("off")

    # Walls
    for (x0, y0, x1, y1) in base.walls:
        ax.add_patch(patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0, color="#2d2d2d", zorder=2))

    # Trajectory trail
    if trajectory and len(trajectory) > 1:
        traj = np.array(trajectory)
        ax.plot(traj[:, 0], traj[:, 1],
                color="royalblue", alpha=0.4, linewidth=1.0, zorder=3)

    # Goal
    gx, gy = base.goal_pos
    ax.add_patch(patches.Circle(
        (gx, gy), base.goal_radius, color="limegreen", alpha=0.85, zorder=4))
    ax.text(gx, gy, "G", ha="center", va="center",
            fontsize=10, fontweight="bold", color="white", zorder=5)

    # Start marker (faint)
    sx, sy = base.start_pos
    ax.add_patch(patches.Circle(
        (sx, sy), 0.2, color="orange", alpha=0.4, zorder=4))

    # Agent
    px, py = base._pos
    ax.add_patch(patches.Circle(
        (px, py), 0.18, color="royalblue", zorder=6))

    ax.set_title(f"pos=({px:.2f}, {py:.2f})  step={base._step_count}", fontsize=9)
    plt.tight_layout()
    plt.pause(0.02)


# ── Run ──────────────────────────────────────────────────────────────
base_env = FourRoomsMaze2D(maze_size=10.0, wall_thickness=0.4,
                           door_width=1.0, step_size=0.1, max_steps=2000)
env = CustomFourRoomsMaze2D(base_env, wind_std=0.02)

fig, ax = plt.subplots(figsize=(6, 6))
plt.ion()

obs, _ = env.reset(seed=42)
traj = [base_env._pos.copy()]

for _ in range(2000):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    traj.append(base_env._pos.copy())
    render_maze2d(env, ax, traj)
    if done or truncated:
        print(f"Done — goal_reached={done}, steps={base_env._step_count}")
        obs, _ = env.reset()
        traj = []

plt.ioff()
plt.show()

### 4. The implementation of A Single Goal is All you Need.

###  $\phi(s,a)$

In [ ]:
# Now here is the NN to train the sampled batch data from the replay buffer, the goal is to learn a dynamics model that predicts s_next from (s,a) pairs.

import torch.nn as nn
import torch.optim as optim 
import torch.nn.functional as F

class StateActionRepresentationModel(nn.Module): # a simple MLP that takes in (s,a) and predicts s_next, use hidden_layers to control the depth of the MLP, and hidden_dim to control the width of the MLP

    def __init__(self, obs_dim, action_dim, hidden_dim=256, output_dim=64, hidden_layers=5,normalise=False): # A single goal paper specifies no normalisation
        super().__init__()
        self.fc1 = nn.Linear(obs_dim + action_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise: # if normalise is True, then we normalise the output to have unit norm, this can help with training stability and convergence, especially when using MSE loss, as it prevents the model from producing arbitrarily large outputs
            x = F.normalize(x, dim=-1)
        return x


 ### $\psi(g)$

In [ ]:
class GoalRepresentationModel(nn.Module): # symmetric counterpart to StateActionRepresentationModel — maps a future/goal state sf to an embedding ψ(sf)

    def __init__(self, obs_dim, hidden_dim=256, hidden_layers=5, output_dim=64, normalise=False): # same architecture choices as phi to keep the embedding space consistent
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, output_dim) # output in the same embedding space as StateActionRepresentationModel so that the dot product phi(s,a)^T psi(sf) is well-defined
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise: # paper specifies no normalisation, but we keep the flag for experimentation
            x = F.normalize(x, dim=-1)
        return x


### InfoNCE + LogSumExp Regularisation Loss Function (Equation 3)

In [ ]:
def contrastive_loss(phi_sa, psi_sf_pos, psi_sf_neg, reg_coef=0.01):
    # Implements the contrastive RL objective from Eq. 3 of the paper:
    #   max  E[ log( e^{phi(s,a)^T psi(sf+)} / (e^{phi(s,a)^T psi(sf+)} + sum_j e^{phi(s,a)^T psi(sf_j-)}) )
    #           - 0.01 * log( sum_j e^{phi(s,a)^T psi(sf_j)} )^2 ]   <- LogSumExp regularisation
    #
    # phi_sa     : (B, d)  state-action embeddings phi(s, a)
    # psi_sf_pos : (B, d)  positive future-state embeddings psi(sf+), sampled via Geometric(1-γ) k steps ahead
    # psi_sf_neg : (B, d)  negative future-state embeddings psi(sf-), sampled uniformly from the replay buffer (marginal distribution)
    #
    # For each anchor phi_sa[i]:
    #   - one positive  : phi_sa[i]^T psi_sf_pos[i]   (col 0 of all_logits)
    #   - B negatives   : phi_sa[i]^T psi_sf_neg[j] for all j  (cols 1..B)
    # This keeps positives and negatives cleanly separated (no accidental positive-as-negative contamination).

    B = phi_sa.shape[0]

    pos_logits = (phi_sa * psi_sf_pos).sum(dim=-1, keepdim=True)  # (B, 1)  one dot-product per positive pair
    neg_logits = phi_sa @ psi_sf_neg.T                            # (B, B)  phi(s_i,a_i)^T psi(sf_j-) for all j

    all_logits = torch.cat([pos_logits, neg_logits], dim=1)  # (B, 1+B): column 0 is the positive
    labels = torch.zeros(B, dtype=torch.long, device=phi_sa.device)  # label 0 = column 0 = positive

    infonce_loss = F.cross_entropy(all_logits, labels)  # minimising this maximises the infoNCE term

    logsumexp = torch.logsumexp(all_logits, dim=-1)  # (B,), log sum_j exp(logit_j)
    reg_loss  = reg_coef * logsumexp.pow(2).mean()   # penalise large LogSumExp values (matches -0.01 * (...) in the maximisation objective)

    return infonce_loss + reg_loss  # minimise this to maximise the paper's contrastive objective


In [ ]:
MIN_STD = 1e-6
LOG_STD_MIN = float(np.log(MIN_STD))
LOG_STD_MAX =  2

class GoalConditionedActor(nn.Module): # goal-conditioned policy pi(a | s, g) — takes (s, g) and outputs a Gaussian action distribution

    def __init__(self, obs_dim, action_dim, hidden_dim=256, hidden_layers=5):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim * 2, hidden_dim)  # concatenate state s and goal g as input
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_mean    = nn.Linear(hidden_dim, action_dim)   # output the mean of the action distribution
        self.log_std_fc = nn.Linear(hidden_dim, action_dim)   # state-dependent log-std head

    def forward(self, obs, goal):
        x = torch.cat([obs, goal], dim=-1)  # (B, obs_dim * 2)
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        mean    = self.fc_mean(x)  # (B, action_dim)
        log_std = self.log_std_fc(x).clamp(LOG_STD_MIN, LOG_STD_MAX)  # (B, action_dim)
        return mean, log_std

    def sample_action(self, obs, goal):
        mean, log_std = self.forward(obs, goal)
        std  = log_std.exp().clamp_min(MIN_STD)
        dist = torch.distributions.Normal(mean, std)

        # Reparameterised sample
        x_t    = dist.rsample()

        # Squash through tanh so actions always lie in (-1, 1)
        action = torch.tanh(x_t)

        # Correct log-prob for the tanh change-of-variables:
        # log π(a|s,g) = log N(x_t; μ, σ) - Σ log(1 - tanh²(x_t))
        log_prob = (dist.log_prob(x_t) - torch.log(1 - action.pow(2) + 1e-6)).sum(dim=-1, keepdim=True)  # (B, 1)
        entropy  = dist.entropy().sum(dim=-1, keepdim=True)  # (B, 1), H(pi(.|s,g))
        return action, log_prob, entropy



In [ ]:
# ── Imports for live visualisation ────────────────────────────────────────────
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ── Hyperparameters ────────────────────────────────────────────────────────────
TOTAL_ENV_STEPS    = 30000   # total environment steps to collect
WARMUP_STEPS       = 2000    # random-action steps before any policy learning
BATCH_SIZE         = 256     # critic / actor batch size
SAMPLES_PER_INSERT = 256     # paper setting: each inserted transition is reused this many times on average
LR                 = 3e-4    # learning rate for critic + actor
INIT_ALPHA         = 0.1     # initial entropy weight before automatic tuning
ALPHA_LR           = 3e-4    # learning rate for entropy-weight tuning
TARGET_ENTROPY     = 0.0     # paper setting so it becomes deterministic
MIN_STD            = 1e-6    # paper setting: minimum actor std
WIND_VARIANCE      = 4e-6    # zero-mean stochastic wind variance added to the car velocity
WIND_STD           = float(np.sqrt(WIND_VARIANCE))
DRAG_COEFF         = 0.01    # light damping so the policy must be robust to persistent drag
LOG_INTERVAL       = 50      # refresh / print every this many gradient-update steps
MAX_HORIZON        = 500 
BUFFER_CAPACITY    = 10000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# ── Single hard goal s* (Alg. 1, line 3) ──────────────────────────────────────
# For MountainCarContinuous the goal is to reach position >= 0.45 with any velocity.
# s* = [0.45, 0.0] is a valid env observation (position=goal_position, velocity=0).
s_star = torch.tensor([0.45, 0.0], device=DEVICE, dtype=torch.float32)

# ── Environment ───────────────────────────────────────────────────────────────
env_train = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_HORIZON)
env_train = CustomMountainCar(env_train, wind_std=WIND_STD, drag_coeff=DRAG_COEFF)

obs_dim    = env_train.observation_space.shape[0]
action_dim = env_train.action_space.shape[0]

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)
print(DEVICE)

phi_model = StateActionRepresentationModel(obs_dim, action_dim, hidden_dim=256, output_dim=64, hidden_layers=5, normalise=False).to(DEVICE)
psi_model = GoalRepresentationModel(obs_dim, hidden_dim=256, output_dim=64, hidden_layers=5, normalise=False).to(DEVICE)
actor     = GoalConditionedActor(obs_dim, action_dim, hidden_dim=256, hidden_layers=5).to(DEVICE)

critic_optimizer = optim.Adam(list(phi_model.parameters()) + list(psi_model.parameters()), lr=LR)
actor_optimizer  = optim.Adam(actor.parameters(), lr=LR)
log_alpha        = torch.tensor(np.log(INIT_ALPHA), dtype=torch.float32, device=DEVICE, requires_grad=True)
alpha_optimizer  = optim.Adam([log_alpha], lr=ALPHA_LR)

# ── History buffers for visualisation ─────────────────────────────────────────
critic_loss_history     = []
actor_loss_history      = []
alpha_loss_history      = []
alpha_history           = []
ep_len_history          = []
entropy_history         = []
log_prob_history        = []
env_steps_at_update     = []  # x-axis: total env steps when each gradient update was performed
goal_similarity_history = []  # mean phi(s,a)^T psi(s*) over each batch — diagnostic for point 6

# ── Helper: critic-only update from replay ────────────────────────────────────
def critic_update_step():
    batch = replay_buffer.sample_future_goal_batch(BATCH_SIZE)

    obs     = batch["obs"]
    actions = batch["actions"]
    goals   = batch["goals"]

    phi_sa    = phi_model(torch.cat([obs, actions], dim=-1))
    psi_g     = psi_model(goals)
    neg_goals = replay_buffer.sample_negative_future_goals(BATCH_SIZE)
    psi_neg   = psi_model(neg_goals)

    critic_loss = contrastive_loss(phi_sa, psi_g, psi_neg)

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    with torch.no_grad():
        psi_sstar = psi_model(s_star.unsqueeze(0).expand(BATCH_SIZE, -1))
        goal_sim  = (phi_sa.detach() * psi_sstar).sum(dim=-1).mean().item()

    return batch, critic_loss, goal_sim

# ── Helper: collect one episode into a dict ────────────────────────────────────
def collect_episode(env, policy_fn):
    """Run one episode. policy_fn(obs_tensor) -> action_np. Returns (ep_dict, ep_steps)."""
    obs_t, _ = env.reset()
    done = False
    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}
    while not done:
        action_np = policy_fn(obs_t)
        next_obs_t, reward, term, trunc, _ = env.step(action_np)
        ep["obs"].append(obs_t.astype(np.float32))
        ep["actions"].append(action_np.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs_t.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))
        obs_t = next_obs_t
        done  = term or trunc
    return ep, len(ep["obs"])

# ── Phase 1: Random-action warmup ─────────────────────────────────────────────
total_env_steps = 0
print(f"Warmup: collecting {WARMUP_STEPS:,} random-action steps …")

def random_policy(obs_t):
    return env_train.action_space.sample()

while total_env_steps < WARMUP_STEPS:
    ep, ep_steps = collect_episode(env_train, random_policy)
    replay_buffer.add_episode(ep)
    total_env_steps += ep_steps

print(f"Warmup done.  Buffer size: {replay_buffer.size:,} transitions  "
      f"({total_env_steps:,} env steps collected).")

# ── Phase 1b: Critic pretraining on warmup replay only ────────────────────────
# This delays actor learning until the representation has already seen the
# broad state coverage from the random warmup data.
critic_pretrain_updates = max(1, int(np.ceil(SAMPLES_PER_INSERT * replay_buffer.size / BATCH_SIZE)))
print(f"Critic warmup: running {critic_pretrain_updates:,} critic-only replay updates before actor learning …")

for warmup_update in range(1, critic_pretrain_updates + 1):
    _, critic_loss, goal_sim = critic_update_step()
    if warmup_update % LOG_INTERVAL == 0 or warmup_update == critic_pretrain_updates:
        print(f"WarmupCritic {warmup_update:>5} / {critic_pretrain_updates:>5} | "
              f"critic_loss: {critic_loss.item():.4f} | goal_sim: {goal_sim:.4f}")

print("Critic warmup complete. Actor updates will start only from the online phase.")

# ── Phase 2: Main training loop ───────────────────────────────────────────────
# Run until we have collected TOTAL_ENV_STEPS total environment interactions.
# After each trajectory insert, perform enough replay updates to match the
# requested samples-per-insert ratio from the paper.
grad_step = 0   # counts online gradient update iterations (for logging and x-axis)
print(f"Training: will run until {TOTAL_ENV_STEPS:,} total env steps …")

s_star_batch = s_star.unsqueeze(0)  # the hard goal

while total_env_steps < TOTAL_ENV_STEPS:

    # ── Collect one trajectory using pi(a | s, g = s*) ───────────────────
    def actor_policy(obs_t):
        obs_tensor = torch.tensor(obs_t, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action, _, _ = actor.sample_action(obs_tensor, s_star_batch)
        action_np = action.squeeze(0).cpu().numpy()
        return action_np

    ep, ep_steps = collect_episode(env_train, actor_policy)
    replay_buffer.add_episode(ep)
    total_env_steps += ep_steps

    updates_this_round = max(1, int(np.ceil(SAMPLES_PER_INSERT * ep_steps / BATCH_SIZE)))

    for _ in range(updates_this_round):
        grad_step += 1

        # ── Critic update (Eq. 3) ─────────────────────────────────────────
        batch, critic_loss, goal_sim = critic_update_step()
        obs = batch["obs"]

        goal_similarity_history.append(goal_sim)

        # ── Actor update (Eq. 4) with automatic entropy tuning ────────────
        sampled_actions, log_prob, entropy = actor.sample_action(obs, s_star_batch.expand(obs.shape[0], -1))

        phi_sa_actor = phi_model(torch.cat([obs, sampled_actions], dim=-1))
        psi_g_actor  = psi_model(s_star_batch.expand(obs.shape[0], -1))
        alpha        = log_alpha.exp()

        # ψ(g) is detached so the actor loss does not update the goal encoder —
        # gradients flow only through φ(s, a_sampled), i.e. through the actor.
        q_values   = (phi_sa_actor * psi_g_actor.detach()).sum(dim=-1, keepdim=True)
        actor_loss = (alpha.detach() * log_prob - q_values).mean()

        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        # Target entropy = 0 means the policy should become increasingly
        # deterministic over training rather than keeping a fixed entropy bonus.
        alpha_loss = (alpha * (-log_prob.detach() - TARGET_ENTROPY)).mean()

        alpha_optimizer.zero_grad()
        alpha_loss.backward()
        alpha_optimizer.step()

        # ── Record metrics ────────────────────────────────────────────────
        critic_loss_history.append(critic_loss.item())
        actor_loss_history.append(actor_loss.item())
        alpha_loss_history.append(alpha_loss.item())
        alpha_history.append(alpha.item())
        ep_len_history.append(ep_steps)
        entropy_history.append(entropy.detach().mean().item())
        log_prob_history.append(log_prob.detach().mean().item())
        env_steps_at_update.append(total_env_steps)

        # ── Live plot every LOG_INTERVAL gradient steps ───────────────────
        if grad_step % LOG_INTERVAL == 0:
            print(f"GradStep {grad_step:>5} | env_steps: {total_env_steps:>7,} | ep_len: {ep_steps:>3} | "
                  f"updates/insert: {updates_this_round:>3} | critic_loss: {critic_loss.item():.4f} | "
                  f"actor_loss: {actor_loss.item():.4f} | alpha: {alpha.item():.6f} | goal_sim: {goal_sim:.4f}")

            clear_output(wait=True)
            fig, axes = plt.subplots(2, 2, figsize=(14, 8))
            axes = axes.flatten()

            axes[0].plot(env_steps_at_update, critic_loss_history, color='steelblue')
            axes[0].set_title("Critic loss (contrastive)")
            axes[0].set_xlabel("Env steps")
            axes[0].set_ylabel("Loss")

            axes[1].plot(env_steps_at_update, actor_loss_history, color='darkorange')
            axes[1].set_title("Actor loss")
            axes[1].set_xlabel("Env steps")
            axes[1].set_ylabel("Loss")

            axes[2].plot(env_steps_at_update, ep_len_history, color='seagreen')
            axes[2].axhline(MAX_HORIZON, color='gray', linestyle='--', linewidth=0.8, label='max horizon')
            axes[2].set_title("Episode length")
            axes[2].set_xlabel("Env steps")
            axes[2].set_ylabel("Steps")
            axes[2].legend()

            axes[3].plot(env_steps_at_update, goal_similarity_history, color='mediumpurple')
            axes[3].axhline(0, color='gray', linestyle='--', linewidth=0.8)
            axes[3].set_title(r"Goal similarity: $\phi(s,a)^\top\psi(s^*)$")
            axes[3].set_xlabel("Env steps")
            axes[3].set_ylabel("Mean dot-product")

            plt.suptitle(f"Training progress — {total_env_steps:,} / {TOTAL_ENV_STEPS:,} env steps", fontsize=12)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

env_train.close()
print("Training complete!")
print(f"Total env steps: {total_env_steps:,} | Online gradient updates: {grad_step:,}")
print(f"Replay buffer size: {replay_buffer.stats()['size']:,}")
print(f"Final alpha: {log_alpha.exp().item():.6f} | Target entropy: {TARGET_ENTROPY:.1f}")



### 5. Post-training summary plots


### 5a. Diagnostic: Network I/O distributions & positive/negative batch example

In [ ]:
# ── Diagnostic: visualise network inputs, outputs, and one contrastive batch ──────
# Run this cell after training (phi_model, psi_model, replay_buffer must exist).
#
# Panel layout:
#   Row 1: distributions of raw network inputs  (state, action, goal)
#   Row 2: distributions of network outputs φ(s,a) and ψ(g)
#   Row 3: concrete single-batch example — pos logit vs neg logits distribution
#          + scatter of φ and ψ in 2-D representation space

import matplotlib.pyplot as plt
import numpy as np
import torch

_DIAG_B = 512   # batch size used for diagnostics
phi_model.eval()
psi_model.eval()

with torch.no_grad():
    # ── Sample a batch with positive future goals ─────────────────────────
    dbatch   = replay_buffer.sample_future_goal_batch(_DIAG_B)
    d_obs    = dbatch["obs"]      # (B, 2)  [position, velocity]
    d_act    = dbatch["actions"]  # (B, 1)  action
    d_goals  = dbatch["goals"]    # (B, 2)  positive future state sf+
    d_neg    = replay_buffer.sample_negative_future_goals(_DIAG_B)  # (B, 2) negatives

    # ── Network outputs ───────────────────────────────────────────────────
    d_phi    = phi_model(torch.cat([d_obs, d_act], dim=-1))  # (B, 2)
    d_psi_p  = psi_model(d_goals)    # (B, 2)  positive goal repr
    d_psi_n  = psi_model(d_neg)      # (B, 2)  negative goal repr

    # ── Logit distributions ───────────────────────────────────────────────
    pos_logits = (d_phi * d_psi_p).sum(dim=-1).cpu().numpy()      # (B,) diagonal
    neg_logits = (d_phi @ d_psi_n.T).cpu().numpy().flatten()      # (B*B,) all cross

    d_obs_np   = d_obs.cpu().numpy()
    d_act_np   = d_act.cpu().numpy()
    d_goals_np = d_goals.cpu().numpy()
    d_neg_np   = d_neg.cpu().numpy()
    d_phi_np   = d_phi.cpu().numpy()
    d_psi_p_np = d_psi_p.cpu().numpy()
    d_psi_n_np = d_psi_n.cpu().numpy()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

# ── Row 0: input distributions ────────────────────────────────────────────────
axes[0, 0].hist(d_obs_np[:, 0], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 0].set_title('Input φ: position (s[0])')
axes[0, 0].set_xlabel('position')

axes[0, 1].hist(d_obs_np[:, 1], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 1].set_title('Input φ: velocity (s[1])')
axes[0, 1].set_xlabel('velocity')

axes[0, 2].hist(d_act_np[:, 0], bins=40, color='coral', edgecolor='white', linewidth=0.3)
axes[0, 2].set_title('Input φ: action (a[0])')
axes[0, 2].set_xlabel('action')

# ── Row 1: network output distributions ───────────────────────────────────────
axes[1, 0].hist(d_phi_np[:, 0], bins=40, color='darkorange', edgecolor='white', linewidth=0.3, label='φ dim-0', alpha=0.7)
axes[1, 0].hist(d_phi_np[:, 1], bins=40, color='gold',       edgecolor='white', linewidth=0.3, label='φ dim-1', alpha=0.7)
axes[1, 0].set_title('Output φ(s,a): both dimensions')
axes[1, 0].legend()

axes[1, 1].hist(d_psi_p_np[:, 0], bins=40, color='seagreen',    edgecolor='white', linewidth=0.3, label='ψ(sf+) dim-0', alpha=0.7)
axes[1, 1].hist(d_psi_p_np[:, 1], bins=40, color='limegreen',   edgecolor='white', linewidth=0.3, label='ψ(sf+) dim-1', alpha=0.7)
axes[1, 1].set_title('Output ψ(positive sf+): both dimensions')
axes[1, 1].legend()

axes[1, 2].hist(d_psi_n_np[:, 0], bins=40, color='mediumpurple', edgecolor='white', linewidth=0.3, label='ψ(sf-) dim-0', alpha=0.7)
axes[1, 2].hist(d_psi_n_np[:, 1], bins=40, color='plum',         edgecolor='white', linewidth=0.3, label='ψ(sf-) dim-1', alpha=0.7)
axes[1, 2].set_title('Output ψ(negative sf-): both dimensions')
axes[1, 2].legend()

# ── Row 2: logit distributions + 2-D scatter ──────────────────────────────────
axes[2, 0].hist(pos_logits, bins=40, color='seagreen',    edgecolor='white', linewidth=0.3, label=f'positive (n={len(pos_logits)})', alpha=0.8)
axes[2, 0].hist(neg_logits, bins=80, color='mediumpurple', edgecolor='white', linewidth=0.2, label=f'negative (n={len(neg_logits)})', alpha=0.5)
axes[2, 0].axvline(np.mean(pos_logits), color='darkgreen',  linestyle='--', linewidth=1.5, label=f'μ_pos={np.mean(pos_logits):.2f}')
axes[2, 0].axvline(np.mean(neg_logits), color='darkviolet', linestyle='--', linewidth=1.5, label=f'μ_neg={np.mean(neg_logits):.2f}')
axes[2, 0].set_title('Logit distributions: pos vs neg\n(gap = contrastive signal)')
axes[2, 0].set_xlabel('φ(s,a) · ψ(g)')
axes[2, 0].legend(fontsize=8)

# 2-D scatter of φ and ψ in representation space
sc_phi = axes[2, 1].scatter(d_phi_np[:, 0],   d_phi_np[:, 1],   c='darkorange',    s=8, alpha=0.5, label='φ(s,a)')
sc_psp = axes[2, 1].scatter(d_psi_p_np[:, 0], d_psi_p_np[:, 1], c='seagreen',      s=8, alpha=0.5, label='ψ(sf+)')
sc_psn = axes[2, 1].scatter(d_psi_n_np[:, 0], d_psi_n_np[:, 1], c='mediumpurple',  s=8, alpha=0.3, label='ψ(sf-)')
# Mark s* goal
with torch.no_grad():
    psi_star_np = psi_model(s_star.unsqueeze(0)).cpu().numpy()
axes[2, 1].scatter(psi_star_np[0, 0], psi_star_np[0, 1], c='red', s=120, marker='*', zorder=5, label='ψ(s*) goal')
axes[2, 1].set_title('Representation space (2-D)\nφ(s,a), ψ(sf+), ψ(sf-), ψ(s*)')
axes[2, 1].set_xlabel('dim 0')
axes[2, 1].set_ylabel('dim 1')
axes[2, 1].legend(fontsize=8)

# Positive goal (sf+) input distribution vs negative goal (sf-) input distribution
axes[2, 2].hist(d_goals_np[:, 0], bins=40, color='seagreen',    edgecolor='white', linewidth=0.3, label='sf+ position', alpha=0.7)
axes[2, 2].hist(d_neg_np[:, 0],   bins=40, color='mediumpurple', edgecolor='white', linewidth=0.3, label='sf- position', alpha=0.7)
axes[2, 2].axvline(0.45, color='red', linestyle='--', linewidth=1.5, label='goal pos=0.45')
axes[2, 2].set_title('Raw position: sf+ vs sf-\n(do positives skew toward goal?)')
axes[2, 2].set_xlabel('position')
axes[2, 2].legend(fontsize=8)

plt.suptitle(
    f'Network I/O Diagnostic  |  buffer={replay_buffer.size:,} transitions  |  '
    f'repr_dim=obs_dim={obs_dim}  |  normalise={phi_model.normalise}',
    fontsize=11
)
plt.tight_layout()
plt.savefig('/tmp/network_io_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

phi_model.train()
psi_model.train()

print(f'\nSummary statistics:')
print(f'  φ(s,a) output  — mean: {d_phi_np.mean():.3f}  std: {d_phi_np.std():.3f}  min: {d_phi_np.min():.3f}  max: {d_phi_np.max():.3f}')
print(f'  ψ(sf+)  output  — mean: {d_psi_p_np.mean():.3f}  std: {d_psi_p_np.std():.3f}  min: {d_psi_p_np.min():.3f}  max: {d_psi_p_np.max():.3f}')
print(f'  ψ(sf-)  output  — mean: {d_psi_n_np.mean():.3f}  std: {d_psi_n_np.std():.3f}  min: {d_psi_n_np.min():.3f}  max: {d_psi_n_np.max():.3f}')
print(f'  pos logits      — mean: {pos_logits.mean():.3f}  std: {pos_logits.std():.3f}')
print(f'  neg logits      — mean: {neg_logits.mean():.3f}  std: {neg_logits.std():.3f}')
print(f'  contrastive gap (μ_pos - μ_neg): {pos_logits.mean() - neg_logits.mean():.3f}  (positive = good)')


In [ ]:
# ── Post-training summary: four-panel static plot ─────────────────────────────
# Smooth a 1-D list with a simple rolling window mean to reduce noise in the curves.
def smooth(values, window=10):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')

steps_full = np.asarray(env_steps_at_update)
smooth_w   = max(1, len(steps_full) // 20) if len(steps_full) > 0 else 1

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ─ Critic loss ─
ax = axes[0, 0]
ax.plot(steps_full, critic_loss_history, color='steelblue', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(critic_loss_history, smooth_w),
        color='steelblue', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.set_title("Critic loss (contrastive, Eq. 3)")
ax.set_xlabel("Env steps")
ax.set_ylabel("Loss")
ax.legend()

# ─ Actor loss ─
ax = axes[0, 1]
ax.plot(steps_full, actor_loss_history, color='darkorange', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(actor_loss_history, smooth_w),
        color='darkorange', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.set_title("Actor loss (Eq. 4)")
ax.set_xlabel("Env steps")
ax.set_ylabel("Loss")
ax.legend()

# ─ Episode length ─
ax = axes[1, 0]
ax.plot(steps_full, ep_len_history, color='seagreen', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(ep_len_history, smooth_w),
        color='seagreen', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.axhline(MAX_HORIZON, color='gray', linestyle='--', linewidth=0.8, label=f'max horizon ({MAX_HORIZON})')
ax.set_title("Episode length")
ax.set_xlabel("Env steps")
ax.set_ylabel("Steps per episode")
ax.legend()
# A shorter episode that terminates (rather than truncates) means the agent reached the goal.

# ─ Policy entropy ─
ax = axes[1, 1]
ax.plot(steps_full, entropy_history, color='mediumpurple', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(entropy_history, smooth_w),
        color='mediumpurple', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.axhline(TARGET_ENTROPY, color='gray', linestyle='--', linewidth=0.8, label=f'target entropy ({TARGET_ENTROPY:.1f})')
ax.set_title("Policy entropy  H(π(·|s, g))")
ax.set_xlabel("Env steps")
ax.set_ylabel("Entropy (nats)")
ax.legend()
# With target entropy = 0, the policy entropy should trend downward toward zero over training.

plt.suptitle("Post-training summary", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



### 6. State-coverage heatmap


In [ ]:
# ── State-coverage heatmap ─────────────────────────────────────────────────────
# Visualise WHERE in (position, velocity) space the agent has been during training.
# Dense regions → well explored; sparse regions → rarely visited.
# If the goal position (0.45) is never reached, the right-hand side will be empty.

all_states = replay_buffer.obs[: replay_buffer.size]   # (N, 2) numpy array

positions  = all_states[:, 0]   # x-axis: car position ∈ [-1.2, 0.6]
velocities = all_states[:, 1]   # y-axis: car velocity ∈ [-0.07, 0.07]

fig, ax = plt.subplots(figsize=(9, 5))

h = ax.hist2d(
    positions, velocities,
    bins=[80, 60],
    range=[[-1.2, 0.6], [-0.07, 0.07]],
    cmap='YlOrRd',
    density=False,
)
cbar = plt.colorbar(h[3], ax=ax)
cbar.set_label("Visit count")

# Goal position marker
ax.axvline(0.45, color='lime', linewidth=1.5, linestyle='--', label='goal position (0.45)')

# Starting region: env resets position uniformly in [-0.6, -0.4]
ax.axvspan(-0.6, -0.4, alpha=0.12, color='dodgerblue', label='reset region')

ax.set_xlabel("Position")
ax.set_ylabel("Velocity")
ax.set_title("State-coverage heatmap (all replay-buffer transitions)")
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()
print(f"Plotted {replay_buffer.size:,} transitions.")


### 7. Final policy rollout — render learned behaviour


In [ ]:
# ── Render the learned policy and save as an animated GIF ─────────────────────
# The policy is always commanded toward the single hard goal s* = [0.45, 0.0].
# We run RENDER_EPISODES episodes, capture rgb_array frames, and produce a GIF
# so it can be viewed in the notebook and shared without a live display.

import imageio.v2 as imageio   # imageio ships with gymnasium as a dependency

RENDER_EPISODES = 3            # how many full rollouts to stitch together
RENDER_FPS      = 30           # frames per second in the output GIF

env_render = gym.make("MountainCarContinuous-v0",
                      render_mode="rgb_array",
                      max_episode_steps=MAX_HORIZON)
env_render = CustomMountainCar(env_render, wind_std=WIND_STD, drag_coeff=DRAG_COEFF)

phi_model.eval()
psi_model.eval()
actor.eval()

all_frames   = []
ep_rewards   = []
ep_positions = []   # track position over time to visualise progress
ep_velocities = []   # track velocity over time to visualise progress

s_star_render = s_star.unsqueeze(0)   # (1, obs_dim)

for ep_i in range(RENDER_EPISODES):
    obs_r, _ = env_render.reset()
    done_r    = False
    ep_rew    = 0.0
    positions_this_ep = []
    velocities_this_ep = []
    while not done_r:
        frame = env_render.render()    # (H, W, 3) uint8
        all_frames.append(frame)

        obs_tensor_r = torch.tensor(obs_r, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action_r, _, _ = actor.sample_action(obs_tensor_r, s_star_render)
        action_r_np = action_r.squeeze(0).cpu().numpy()
        action_r_np = np.clip(action_r_np, env_render.action_space.low, env_render.action_space.high)

        obs_r, rew_r, term_r, trunc_r, _ = env_render.step(action_r_np)
        ep_rew += rew_r
        positions_this_ep.append(obs_r[0])
        velocities_this_ep.append(obs_r[1])
        done_r = term_r or trunc_r

    ep_rewards.append(ep_rew)
    ep_positions.append(positions_this_ep)
    ep_velocities.append(velocities_this_ep)
    print(f"  Rollout {ep_i+1}: {len(positions_this_ep)} steps | "
          f"total reward: {ep_rew:.2f} | "
          f"max position: {max(positions_this_ep):.3f}")

env_render.close()
phi_model.train()
psi_model.train()
actor.train()

gif_path = "/tmp/policy_rollout.gif"
imageio.mimsave(gif_path, all_frames, fps=RENDER_FPS, loop=0)
print(f"\nSaved GIF ({len(all_frames)} frames) → {gif_path}")

# Display the GIF inline (works in Jupyter Lab / Notebook)
from IPython.display import Image
Image(gif_path)




### 8. Per-rollout position trajectory plot


In [ ]:
# ── Position over time for each rendered rollout ──────────────────────────────
# Helps diagnose whether the agent builds up enough momentum to escape the valley.

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

colours = plt.cm.tab10.colors

# --- Position subplot ---
for i, pos_traj in enumerate(ep_positions):
    ax1.plot(pos_traj, color=colours[i % len(colours)],
             linewidth=1.5, label=f'Rollout {i+1}  (reward={ep_rewards[i]:.1f})')

ax1.axhline(0.45,  color='lime', linestyle='--', linewidth=1.2, label='goal (0.45)')
ax1.axhline(-0.52, color='gray', linestyle=':',  linewidth=0.8, label='valley bottom (≈-0.52)')
ax1.set_ylabel("Car position")
ax1.set_title("Learned policy — position over time (goal = 0.45)")
ax1.legend(fontsize=9)
ax1.set_ylim(-1.25, 0.65)

# --- Velocity subplot ---
for i, vel_traj in enumerate(ep_velocities):
    ax2.plot(vel_traj, color=colours[i % len(colours)],
             linewidth=1.5, label=f'Rollout {i+1}')

ax2.axhline(0,  color='gray', linestyle=':', linewidth=0.8, label='zero velocity')
ax2.set_xlabel("Timestep")
ax2.set_ylabel("Car velocity")
ax2.set_title("Learned policy — velocity over time")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
